# 06 — Hourly Resampling

**Purpose:** Resample all EnFa signals from ~20-second raw resolution to hourly Parquet.

Corresponds to `scripts/07_resample_hourly.py` — same logic, step by step.

**Aggregation rules (critical — do not change without reason):**

| Signal type | Rule | Why |
|---|---|---|
| Power (kW), temperature, SOC, voltage | `MEAN` | Average over the hour |
| Cumulative energy/gas counters (`greal_E_*`) | `MAX − MIN` (delta) | Ever-increasing totals — delta gives energy that hour |
| Defrost duration (`*AbtauSek`) | `SUM` | Total seconds of defrost in that hour |
| Setpoints (`V_real*`, event-driven) | `LAST` + forward-fill | Only written on change; ffill propagates last known value |

Output: `data/processed/hourly.parquet` (~200 MB) — one UTC hour per row, one column per signal.
All notebooks 07+ load this Parquet instead of the raw 40 GB.

In [1]:
import sys
import csv
import time
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from zoro_eda.config import load_config
from zoro_eda.paths import resolve_paths

cfg   = load_config()
paths = resolve_paths(cfg=cfg)

print(f"Project root : {paths.root}")
print(f"Raw data     : {paths.raw_data}")
print(f"Processed    : {paths.processed}")

Project root : C:\Users\dellg\OneDrive\Documents\ZE
Raw data     : C:\Users\dellg\OneDrive\Documents\ZE\data
Processed    : C:\Users\dellg\OneDrive\Documents\ZE\data\processed


## Step 1: Aggregation rule logic

`resolve_aggregation()` maps each signal's category to its correct aggregation type.

Test it on a few signals to confirm it assigns the right rules.

In [3]:
EXCLUDE_STEMS = frozenset({"A", "_value", "pilot"})

_SETPOINT_CATS = frozenset({
    "hp_setpoint", "hvac_setpoint", "night_setback", "dhw_setpoint",
    "battery_setpoint", "storage_setpoint", "heating_curve",
    "control_param", "timer_schedule",
})

def resolve_aggregation(category: str, signal_name: str) -> str:
    cat  = category.lower().strip()
    name = signal_name.lower()
    if cat in ("exclude",) or signal_name in EXCLUDE_STEMS:
        return "skip"
    if any(k in cat  for k in ("energy","gas","volume","counter","pulse","dint")): return "delta"
    if signal_name.startswith("dint"):                                              return "delta"
    if "defrost" in cat or "abtau" in cat or "abtau" in name:                      return "sum"
    if cat in _SETPOINT_CATS:                                                       return "last"
    if any(k in cat  for k in ("setpoint","setback","curve","schedule","timer","param")): return "last"
    return "mean"

# Quick self-test
tests = [
    ("greal_E_WP",          "energy_meter"),
    ("greal_WP1AbtauSek",   "hp_defrost"),
    ("V_real_maxVorlaufTemp","hp_setpoint"),
    ("real_BatterieLeistung","battery_power"),
    ("greal_BatterieLadeZustand", "battery_soc"),
]
print(f"{'Signal':<35} {'Category':<20} {'Agg'}")
print("-" * 62)
for sig, cat in tests:
    print(f"{sig:<35} {cat:<20} {resolve_aggregation(cat, sig)}")

Signal                              Category             Agg
--------------------------------------------------------------
greal_E_WP                          energy_meter         delta
greal_WP1AbtauSek                   hp_defrost           sum
V_real_maxVorlaufTemp               hp_setpoint          last
real_BatterieLeistung               battery_power        mean
greal_BatterieLadeZustand           battery_soc          mean


## Step 2: Check aggregation assignments for all signals

Load the classification from notebook 04 and show which aggregation type
each signal will receive. This is a good sanity check before running the full resample.

In [4]:
def load_category_map(reports_dir: Path) -> dict:
    path = reports_dir / "signal_classification.csv"
    if not path.exists():
        print("WARNING: signal_classification.csv not found — all signals will use 'mean'")
        return {}
    mapping = {}
    with open(path, encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            stem = row.get("signal_name", "").strip()
            cat  = row.get("category",    "").strip()
            excl = row.get("exclude",     "").strip().lower()
            if stem:
                mapping[stem] = "EXCLUDE" if excl == "true" else cat
    return mapping

category_map = load_category_map(paths.reports)

# Show aggregation assignment breakdown
from collections import Counter
assignments = {s: resolve_aggregation(c, s) for s, c in category_map.items()}
agg_counts  = Counter(assignments.values())
print("Aggregation type breakdown:")
for agg, n in sorted(agg_counts.items()):
    print(f"  {agg:<8}  {n} signals")

print()
print("Sample — delta signals (cumulative energy counters):")
delta_sigs = [s for s, a in assignments.items() if a == "delta"][:8]
for s in delta_sigs:
    print(f"  {s}")

Aggregation type breakdown:
  delta     57 signals
  last      38 signals
  mean      131 signals
  skip      3 signals
  sum       4 signals

Sample — delta signals (cumulative energy counters):
  dintVolGasVerbrauchBhkwImp1
  dintVolGasVerbrauchBhkwImp2
  greal_AZ_WP_Energie
  greal_EK_WMZ_WWout
  greal_E_BatterieAbgabe
  greal_E_BatterieLaden
  greal_E_Gebaeude
  greal_E_GebaeudeNord


## Step 3: Demonstrate aggregation on one signal

**Change `SIGNAL` and re-run** to see raw rows vs hourly-aggregated result side by side.

Try `greal_E_WP` (cumulative energy counter → delta) or
`V_real_maxVorlaufTemp` (setpoint → last).

In [5]:
# ---- change this ----
SIGNAL = "greal_E_WP"
# ----------------------

csv_path = paths.raw_data / f"{SIGNAL}.csv"
cat      = category_map.get(SIGNAL, "unknown")
agg      = resolve_aggregation(cat, SIGNAL)

print(f"Signal: {SIGNAL}  Category: {cat}  →  Aggregation: {agg}\n")

# Show first 6 raw rows
RAW = '''
SELECT TRY_CAST(_time AS TIMESTAMPTZ) AS ts, TRY_CAST(_value AS DOUBLE) AS raw_value
FROM read_csv('{path}', delim=';', header=true, ignore_errors=true)
ORDER BY ts LIMIT 6
'''
print("Raw rows (20-second resolution):")
print(duckdb.sql(RAW.format(path=csv_path.as_posix())).df().to_string(index=False))

# Show first 6 hourly-aggregated rows
AGG_SQL = {
    "mean":  "AVG(TRY_CAST(_value AS DOUBLE))",
    "delta": "GREATEST(MAX(TRY_CAST(_value AS DOUBLE)) - MIN(TRY_CAST(_value AS DOUBLE)), 0.0)",
    "sum":   "SUM(TRY_CAST(_value AS DOUBLE))",
    "last":  "ARG_MAX(TRY_CAST(_value AS DOUBLE), TRY_CAST(_time AS TIMESTAMPTZ))",
}
expr = AGG_SQL.get(agg, AGG_SQL["mean"])
HOURLY = f'''
SELECT DATE_TRUNC('hour', TRY_CAST(_time AS TIMESTAMPTZ)) AS hour_utc,
       ROUND({expr}, 4) AS value
FROM read_csv('{{path}}', delim=';', header=true, ignore_errors=true)
WHERE TRY_CAST(_value AS DOUBLE) IS NOT NULL
GROUP BY 1 ORDER BY 1 LIMIT 6
'''
print(f"\nHourly ({agg}):")
print(duckdb.sql(HOURLY.format(path=csv_path.as_posix())).df().to_string(index=False))

Signal: greal_E_WP  Category: hp_energy  →  Aggregation: delta

Raw rows (20-second resolution):
                       ts  raw_value
2022-12-22 16:24:12+02:00   0.992991
2022-12-22 16:24:32+02:00   0.993991
2022-12-22 16:24:52+02:00   0.993991
2022-12-22 16:25:12+02:00   0.994991
2022-12-22 16:25:32+02:00   0.995991
2022-12-22 16:25:52+02:00   0.995991

Hourly (delta):
                 hour_utc  value
2022-12-22 16:00:00+02:00  0.079
2022-12-22 17:00:00+02:00  0.137
2022-12-22 18:00:00+02:00  0.138
2022-12-22 19:00:00+02:00  1.394
2022-12-22 20:00:00+02:00  0.143
2022-12-22 21:00:00+02:00  0.148


## Step 4: Resample all signals to hourly

**This takes 30-60 minutes.** Run once. If `hourly.parquet` already exists, skip.

Each signal is read by DuckDB and reduced to one value per UTC hour.
The results are collected as pandas Series and joined into a wide DataFrame.

In [6]:
out_parquet = paths.processed / "hourly.parquet"
paths.processed.mkdir(parents=True, exist_ok=True)

if out_parquet.exists():
    print(f"Already exists: {out_parquet}")
    print("Delete it and re-run to regenerate.")
else:
    EXCLUDE_STEMS_RUN     = frozenset({"A", "_value", "pilot"})
    SNAPSHOT_THRESHOLD    = 20

    csv_files_all = sorted(
        f for f in paths.raw_data.iterdir()
        if f.is_file() and f.suffix.lower() == ".csv"
        and f.stem not in EXCLUDE_STEMS_RUN
        and category_map.get(f.stem, "") != "EXCLUDE"
    )
    print(f"Resampling {len(csv_files_all)} signals → {out_parquet}")

    # DuckDB SQL templates — one per aggregation type
    AGG_TEMPLATES = {
        "mean": '''
            SELECT DATE_TRUNC('hour', TRY_CAST(_time AS TIMESTAMPTZ)) AS hour_utc,
                   AVG(TRY_CAST(_value AS DOUBLE)) AS value
            FROM read_csv('{path}', delim=';', header=true, ignore_errors=true)
            WHERE TRY_CAST(_value AS DOUBLE) IS NOT NULL
            GROUP BY 1 ORDER BY 1''',
        "delta": '''
            SELECT DATE_TRUNC('hour', TRY_CAST(_time AS TIMESTAMPTZ)) AS hour_utc,
                   GREATEST(MAX(TRY_CAST(_value AS DOUBLE))
                            - MIN(TRY_CAST(_value AS DOUBLE)), 0.0) AS value
            FROM read_csv('{path}', delim=';', header=true, ignore_errors=true)
            WHERE TRY_CAST(_value AS DOUBLE) IS NOT NULL
            GROUP BY 1 ORDER BY 1''',
        "sum": '''
            SELECT DATE_TRUNC('hour', TRY_CAST(_time AS TIMESTAMPTZ)) AS hour_utc,
                   SUM(TRY_CAST(_value AS DOUBLE)) AS value
            FROM read_csv('{path}', delim=';', header=true, ignore_errors=true)
            WHERE TRY_CAST(_value AS DOUBLE) IS NOT NULL
            GROUP BY 1 ORDER BY 1''',
        "last": '''
            SELECT DATE_TRUNC('hour', TRY_CAST(_time AS TIMESTAMPTZ)) AS hour_utc,
                   ARG_MAX(TRY_CAST(_value AS DOUBLE),
                           TRY_CAST(_time AS TIMESTAMPTZ)) AS value
            FROM read_csv('{path}', delim=';', header=true, ignore_errors=true)
            WHERE TRY_CAST(_value AS DOUBLE) IS NOT NULL
            GROUP BY 1 ORDER BY 1''',
    }

    con = duckdb.connect()
    con.execute("PRAGMA threads=4")

    series_list    = []
    setpoint_cols  = []
    skipped, t0    = 0, time.time()

    for i, fpath in enumerate(csv_files_all):
        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{len(csv_files_all)}  elapsed={time.time()-t0:.0f}s")
        stem = fpath.stem
        cat  = category_map.get(stem, "unknown")
        agg  = resolve_aggregation(cat, stem)
        if agg == "skip":
            skipped += 1
            continue
        try:
            # Skip commissioning snapshots
            count_sql = f"SELECT COUNT(*) FROM read_csv('{fpath.as_posix()}', delim=';', header=true, ignore_errors=true)"
            rc = con.execute(count_sql).fetchone()[0]
            if rc < SNAPSHOT_THRESHOLD:
                skipped += 1
                continue
            sql = AGG_TEMPLATES[agg].format(path=fpath.as_posix())
            df  = con.execute(sql).df()
            if df.empty:
                skipped += 1
                continue
            df["hour_utc"] = pd.to_datetime(df["hour_utc"], utc=True)
            s = df.set_index("hour_utc")["value"].rename(stem)
            series_list.append(s)
            if agg == "last":
                setpoint_cols.append(stem)
        except Exception as exc:
            print(f"  WARN {stem}: {exc}")
            skipped += 1

    con.close()
    print(f"\nCollected {len(series_list)} signals  |  Skipped {skipped}")
    print("Assembling wide DataFrame...")

Resampling 230 signals → C:\Users\dellg\OneDrive\Documents\ZE\data\processed\hourly.parquet
  25/230  elapsed=34s
  50/230  elapsed=86s
  75/230  elapsed=137s
  100/230  elapsed=176s
  125/230  elapsed=240s
  150/230  elapsed=286s
  175/230  elapsed=336s
  200/230  elapsed=372s
  225/230  elapsed=373s

Collected 215 signals  |  Skipped 15
Assembling wide DataFrame...


## Step 5: Assemble wide DataFrame and forward-fill setpoints

`pd.concat` joins all hourly Series on the shared UTC time index.

Setpoint columns get forward-filled because they are event-driven —
only written when the value changes. Without ffill, most hours are NaN.

In [ ]:
# This cell runs immediately after Step 4 (series_list must exist in memory)
wide = pd.concat(series_list, axis=1)
wide.index = pd.DatetimeIndex(wide.index, name="hour_utc")
wide.sort_index(inplace=True)

# Forward-fill setpoint columns
cols_present = [c for c in setpoint_cols if c in wide.columns]
if cols_present:
    wide[cols_present] = wide[cols_present].ffill()
    print(f"Forward-filled {len(cols_present)} setpoint columns")

print(f"Wide DataFrame : {wide.shape[0]:,} rows × {wide.shape[1]} columns")
print(f"Time range     : {wide.index.min()}  →  {wide.index.max()}")
print(f"Memory         : {wide.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()
print(wide.iloc[:3, :6])

## Step 6: Save to Parquet and verify

In [ ]:
table = pa.Table.from_pandas(wide, preserve_index=True)
pq.write_table(table, out_parquet, compression="snappy")

size_mb = out_parquet.stat().st_size / 1e6
print(f"Saved: {out_parquet}")
print(f"Size : {size_mb:.1f} MB")

# Reload to verify round-trip
df_check = pd.read_parquet(out_parquet)
print(f"Reload check : {df_check.shape[0]:,} rows × {df_check.shape[1]} columns  ✓")
print(f"Time range   : {df_check.index.min()}  →  {df_check.index.max()}")
print(f"\nColumns (first 10):")
for col in df_check.columns[:10]:
    print(f"  {col}")

## Key findings

After running Steps 4-6:
- `hourly.parquet` is the single source of truth for all EDA and modeling notebooks
- Delta signals (`greal_E_*`) now represent actual energy consumed per hour (not running totals)
- Setpoint columns are fully populated via ffill — no NaN gaps
- Notebook 07 (full EDA) loads this Parquet instantly

Check the Parquet shape — expect roughly 30,000 rows (3.5 years × 8,760 hours/year ≈ 30,660)
and ~200 columns.